<div dir="rtl" style="text-align:right">
<h1 style="text-align:right">سه معماری را با مسیر اطلاعات تشخیص دهید</h1>
<p style="text-align:right">درس 48 از 92 · <bdi dir="ltr">Transformer</bdi> چه فرقی با <bdi dir="ltr">GPT</bdi> دارد؟ · <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">42-families</code></p>
<p style="text-align:right"><a target="_self" href="http://127.0.0.1:8000/part-06/chapter-04/42-families.html">📖 بازگشت به همین درس</a></p>
<p style="text-align:right"><bdi dir="ltr">Self-Attention</bdi> دوطرفه، <bdi dir="ltr">Self-Attention</bdi> علّی و <bdi dir="ltr">Cross-attention</bdi> مستطیلی را با عدد مقایسه کنید.</p><p style="text-align:right">پیش‌نیاز: <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">Q/K/V</code>، <bdi dir="ltr">Mask</bdi> و شکل ضرب <bdi dir="ltr">Attention</bdi> را بشناسید؛ منبع و هدف دو دنبالهٔ متفاوت‌اند.</p>
<p style="text-align:right">زمان یادگیری درس همراه با همین دفتر: حدود ۳۰–۵۵ دقیقه. زمان دفتر دوباره به زمان درس اضافه نمی‌شود؛ نصب و تمرین اختیاری جداست.</p>
<p style="text-align:right">این دفتر نیمهٔ عملی درس است. مثال‌ها آمادهٔ اجرا هستند؛ دو <bdi dir="ltr">Cell</bdi> با برچسب <bdi dir="ltr">TODO</bdi> را خودتان کامل کنید. پیام <bdi dir="ltr">INCOMPLETE</bdi> یعنی هنوز چیزی ننوشته‌اید، نه اینکه پاسخ درست است. جواب مرجع در این دفتر پنهان نشده است.</p>
<p style="text-align:right">از بالا به پایین اجرا کنید. پس از تغییر هر تابع، <bdi dir="ltr">Cell</bdi> آن و سپس <bdi dir="ltr">Cell</bdi> آزمون را دوباره اجرا کنید. برای بررسی نهایی، از منوی <code style="direction:ltr;text-align:left;unicode-bidi:isolate">Kernel → Restart Kernel and Run All Cells</code> استفاده کنید.</p>
</div>

In [ ]:
from pathlib import Path
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">قبل از اجرا، پیش‌بینی کنید</h2>
<p style="text-align:right">منبع ۵ موقعیت و خروجی فعلی ۳ موقعیت دارد. چرا بستن ستون‌های بعد از قطر در <bdi dir="ltr">Cross-attention</bdi>، بخشی از منبعِ ازپیش‌موجود را بی‌دلیل پنهان می‌کند؟</p>
</div>

<div dir="rtl" style="text-align:right"><p style="text-align:right">پیش‌بینی من: …</p></div>

In [ ]:
import math
import torch
torch.set_num_threads(1)
torch.manual_seed(17)
source = torch.tensor([[1.,0.],[0.,1.],[1.,1.],[-1.,0.],[0.,-1.]])
target = torch.tensor([[1.,0.],[0.,1.],[1.,1.]])
print('source/target lengths:',len(source),len(target))

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">این بار شما کد بنویسید</h2>
<p style="text-align:right">تابع <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">family_attention(q,k,v,kind)</code> زوج <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">(weights,output)</code> برگرداند. <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">kind</code> برابر <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">encoder</code> یا <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">decoder</code> یا <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">cross</code> است. فقط در <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">decoder</code> جدول مربعی را علّی کنید؛ در دو حالت دیگر همهٔ <bdi dir="ltr">Key</bdi>ها مجازند. تعداد ویژگی‌های <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">q</code> و <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">k</code> برابر است، ولی تعداد سطرهایشان می‌تواند متفاوت باشد.</p>
</div>

In [ ]:
def family_attention(q, k, v, kind):
    # TODO
    return None

In [ ]:
def test_exercise():
    result = family_attention(target,source,source,'cross')
    if result is None: return False
    w,out = result
    assert w.shape == (3,5) and out.shape == (3,2)
    torch.testing.assert_close(w,(target@source.T/math.sqrt(2)).softmax(-1))
    torch.testing.assert_close(out,w@source)
    we,_ = family_attention(source,source,source,'encoder')
    assert we.shape == (5,5) and (we > 0).all()
    wd,_ = family_attention(target,target,target,'decoder')
    assert torch.count_nonzero(wd.triu(1)) == 0
    torch.testing.assert_close(wd.sum(-1),torch.ones(3))
    zq,zk,zv = torch.zeros(2,4),torch.zeros(3,4),torch.tensor([[1.],[2.],[6.]])
    torch.testing.assert_close(family_attention(zq,zk,zv,'cross')[1],torch.full((2,1),3.))
    return True

exercise_complete = test_exercise()
print("PASS" if exercise_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">فقط یک عامل را تغییر دهید</h2>
<p style="text-align:right">فقط طول منبع را از ۵ به ۴ کاهش دهید؛ <bdi dir="ltr">Query</bdi>های خروجی ثابت بمانند. تعداد سطرهای <bdi dir="ltr">Cross-attention</bdi> از هدف و تعداد ستون‌ها از منبع می‌آیند.</p>
</div>

In [ ]:
for length in (5,4):
    weights = (target@source[:length].T/math.sqrt(2)).softmax(-1)
    print('source length, weights/output shapes:',length,weights.shape,(weights@source[:length]).shape)

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">خرابی را پیدا کنید</h2>
<p style="text-align:right">کد خراب <bdi dir="ltr">Mask</bdi> علّیِ هدف را به جدول <bdi dir="ltr">Cross-attention</bdi> تعمیم می‌دهد. تابع <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">cross_weights(q,k)</code> را اصلاح کنید؛ تمام منبع در این مثال از ابتدا موجود است.</p>
</div>

In [ ]:
scores = target@source.T/math.sqrt(2)
wrong = scores.masked_fill(~torch.ones(3,5,dtype=torch.bool).tril(),float('-inf')).softmax(-1)
print('wrong: source positions hidden from first target:',wrong[0])

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">اصلاح را خودتان بنویسید</h2>
<p style="text-align:right">علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def cross_weights(q, k):
    # TODO
    return None

In [ ]:
def test_repair():
    result = cross_weights(target,source)
    if result is None: return False
    torch.testing.assert_close(result,(target@source.T/math.sqrt(2)).softmax(-1))
    assert (result > 0).all()
    torch.testing.assert_close(cross_weights(torch.zeros(2,3),torch.zeros(4,3)),torch.full((2,4),0.25))
    return True

repair_complete = test_repair()
print("PASS" if repair_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">در <bdi dir="ltr">Mini-GPT</bdi> کجا به کار می‌آید؟</h2>
<p style="text-align:right"><code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">MiniGPT</code> از حالت <bdi dir="ltr">Decoder-only</bdi> استفاده می‌کند: در هر بلوک <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">Q/K/V</code> از یک جریان‌اند و <bdi dir="ltr">Mask</bdi> علّی داریم. پروژه <bdi dir="ltr">Encoder</bdi> یا <bdi dir="ltr">Cross-attention</bdi> جدا ندارد؛ تابع کوچک این دفتر صرفاً برای مقایسهٔ خانواده‌هاست.</p>
</div>

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">با زبان خودتان توضیح دهید</h2>
<p style="text-align:right">اگر کسی <bdi dir="ltr">Transformer</bdi> را مساوی <bdi dir="ltr">GPT</bdi> بداند، کدام تفاوتِ قابل مشاهده در این دفتر را از دست داده است؟</p>
</div>
<div dir="rtl" style="text-align:right"><p style="text-align:right">پیش‌بینی و مشاهدهٔ من: …</p><p style="text-align:right">علت خرابی و اصلاح من: …</p></div>

<div dir="rtl" style="text-align:right"><p style="text-align:right"><a target="_self" href="http://127.0.0.1:8000/part-06/chapter-04/42-families.html">بازگشت به درس و ادامهٔ مسیر</a> · <a target="_self" href="http://127.0.0.1:8000/answers/42-families.html#lab-solution">فقط پس از تلاش: راه‌حل مرجع آزمایشگاه</a></p></div>